<div style="background: linear-gradient(135deg, #1e3a8a, #3b82f6); padding: 2.5rem; border-radius: 16px; color: white; text-align: center; box-shadow: 0 4px 20px rgba(30,58,138,0.25);">
  <h1 style="font-size: 3rem; margin: 0; font-weight: 800; font-family: 'Outfit', sans-serif;">&#x1F697; Drive Wise</h1>
  <p style="font-size: 1.3rem; margin: 0.6rem 0 0 0; opacity: 0.9; font-weight: 300;">Advanced Metadata-Aware Automotive RAG Assistant</p>
  <p style="font-size: 0.95rem; margin: 0.5rem 0 0 0; opacity: 0.75;">Powered by Google Gemini 2.5 &middot; Two-Stage Hybrid Retrieval & Generative Re-Ranking &middot; Persistent Logging & Metrics</p>
</div>

---

## What This Notebook Accomplishes

This notebook implements a production-grade **two-stage Retrieval-Augmented Generation (RAG)** system for querying car brochures, fully satisfying the DriveWise Architecture Specifications:

1. **Document Version Metadata**: Extracts document version (e.g. publication date, model year) using Gemini from the brochure first page and attaches it as chunk-level metadata to enable tracking.
2. **Two-Stage Re-Ranking**: Implements a distinct second-stage generative re-ranker. Stage 1 retrieves candidates using hybrid scoring (semantic similarity + exact keyword overlap). Stage 2 uses a generative model to re-order the top candidates by query relevance before generation.
3. **RAG Evaluation (LLM-as-a-Judge)**: Automatically evaluates three vital quality metrics—**Faithfulness**, **Context Relevance**, and **Answer Correctness**—for every generated answer.
4. **Persistent Logging & Monitoring**: Writes queries, latencies, failure flags, retrieved chunks, and evaluation metrics into a persistent SQLite database (`logs/notebook_query_logs.db`) for tracking quality over time.
5. **Standalone Streamlit Web Interface**: Includes execution scripts to run and tunnel the premium Streamlit dashboard web interface directly from Colab.

> **Vector Database Note**: For maximum simplicity, portability, and zero-dependency runtime in Colab, this demo implements a serverless vector database using an in-memory python list index persisted to a clean JSON file on disk. In a production environment, this list can be drop-in replaced with ChromaDB or FAISS.

## Step 1 — Install Dependencies

Installs all packages needed for parsing, embedding, generating, UI widgets, and dashboard graphing.

In [ ]:
!pip install -q google-generativeai pypdf numpy ipywidgets pandas matplotlib
print("All packages installed!")

## Step 2 — Configure Your Gemini API Key

Configure your Google Gemini API Key. 
**On Colab**: Click the **Secrets** icon (left sidebar key icon) &rarr; add `GOOGLE_API_KEY`.

In [ ]:
import os, json, time, hashlib, sqlite3
from datetime import datetime
import numpy as np
import google.generativeai as genai
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pypdf import PdfReader

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
    print("API key loaded from Colab Secrets.")
except Exception:
    pass

if not api_key:
    api_key = os.environ.get('GOOGLE_API_KEY')
    if api_key:
        print("API key loaded from environment variable.")

if not api_key:
    api_key = "YOUR_GEMINI_API_KEY_HERE"  # Paste here if needed
    print("WARNING: No API key found. Set GOOGLE_API_KEY in Colab Secrets.")

genai.configure(api_key=api_key)
print("Gemini API configured!")

def generate_content_with_retry(model, prompt, generation_config=None, max_retries=5):
    delay = 10
    for attempt in range(max_retries):
        try:
            if generation_config:
                return model.generate_content(prompt, generation_config=generation_config)
            else:
                return model.generate_content(prompt)
        except Exception as e:
            err_str = str(e).lower()
            if any(term in err_str for term in ["429", "quota", "limit", "exhausted"]):
                if attempt == max_retries - 1:
                    raise e
                print(f"  Rate limit hit. Retrying in {delay}s (Attempt {attempt+1}/{max_retries})...")
                time.sleep(delay)
                delay *= 2
            else:
                raise e


## Step 3 — Download Sample Brochures & Index with Version Metadata

Downloads sample PDF brochures and chunks them. During chunking, Gemini extracts the car brand, model, and document version from the first page text to include them in the chunk metadata.

In [ ]:
import urllib.request

SECTION_KEYWORDS = {
    "Engine & Performance": ["engine","torque","power","gearbox","transmission","hp","ps","cc",
        "cylinder","performance","speed","acceleration","manual","automatic","dct","cvt","bhp","rpm"],
    "Mileage & Fuel Efficiency": ["mileage","fuel economy","fuel efficiency","kmpl","km/l",
        "consumption","hybrid","electric range","efficiency","co2","emissions","arai","wltp"],
    "Safety": ["safety","airbag","abs","ebd","esc","brake","crash test","ncap","adas",
        "lane assist","isofix","hill assist","esp","traction control","rear view camera","tpms"],
    "Dimensions": ["dimensions","length","width","height","wheelbase","ground clearance",
        "boot space","weight","capacity","turning radius","fuel tank","kerb weight","mm"],
    "Interior & Comfort": ["interior","comfort","seat","upholstery","climate control","ac",
        "sunroof","steering","cabin","leather","ventilated","ambient lighting","armrest","cruise control"],
    "Infotainment & Connectivity": ["infotainment","screen","display","apple carplay","android auto",
        "bluetooth","speakers","audio","navigation","connected car","usb","voice command","touchscreen"],
}

def classify_section(text):
    scores = {sec: 0 for sec in SECTION_KEYWORDS}
    tl = text.lower()
    for sec, kws in SECTION_KEYWORDS.items():
        for kw in kws:
            scores[sec] += tl.count(kw)
    best = max(scores, key=scores.get)
    return best if scores[best] >= 2 else "General Specifications"

def extract_metadata(filepath, first_page_text):
    """
    Identifies car brand, model, and document version using filename
    and Gemini LLM fallback to parse the first page text.
    """
    filename = os.path.basename(filepath)
    name_without_ext = os.path.splitext(filename)[0]
    
    brand, model, version = "Unknown", name_without_ext.title(), "1.0"
    for sep in ["_", "-"]:
        if sep in name_without_ext:
            parts = name_without_ext.split(sep)
            brand = parts[0].strip().title()
            model = " ".join(parts[1:]).strip().title()
            break
            
    try:
        prompt = f"""
        Extract the car brand, model, and brochure document version from the following text of the first page of the car brochure.
        Look for indicators of document version such as a version number (e.g. v1.1, version 2.0), model year (e.g. MY24, MY2023), or publication month/year (e.g. 10/2023, July 2022).
        
        Text:
        ---
        {first_page_text[:2000]}
        ---
        Respond with ONLY a valid JSON object in this format (no markdown blocks, just raw JSON text):
        {{"brand": "BrandName", "model": "ModelName", "version": "VersionInfo"}}
        If you cannot extract the version, return "1.0" as the default version.
        """
        model_gen = genai.GenerativeModel("models/gemini-flash-latest")
        response = generate_content_with_retry(model_gen, prompt)
        text = response.text.strip()
        if text.startswith("```json"):
            text = text[7:]
        if text.endswith("```"):
            text = text[:-3]
        text = text.strip()
        data = json.loads(text)
        brand_extracted = brand if brand not in ["Unknown", "Unknown Brand"] else data.get("brand", "").strip().title()
        model_extracted = model if brand not in ["Unknown", "Unknown Brand"] else data.get("model", "").strip().title()
        version_extracted = data.get("version", "1.0").strip()
        if not brand_extracted or brand_extracted.lower() in ["unknown", "unknown brand"]:
            brand_extracted = brand
        if not model_extracted or model_extracted.lower() in ["unknown", "unknown model"]:
            model_extracted = model
        return brand_extracted, model_extracted, version_extracted
    except Exception:
        return brand, model, version

def chunk_pdf(filepath):
    reader = PdfReader(filepath)
    first_page_text = reader.pages[0].extract_text() if len(reader.pages) > 0 else ""
    brand, model, version = extract_metadata(filepath, first_page_text)
    chunks = []
    for page_idx, page in enumerate(reader.pages):
        text = page.extract_text()
        if not text or not text.strip():
            continue
        paras = text.split("\n\n")
        current, current_len = [], 0
        for para in paras:
            para = para.strip()
            if not para:
                continue
            if current_len + len(para) > 800 and current:
                chunk_text = "\n".join(current)
                chunks.append({
                    "text": chunk_text, "brand": brand, "model": model, "version": version,
                    "section": classify_section(chunk_text),
                    "page": page_idx + 1, "source_file": os.path.basename(filepath)
                })
                current, current_len = [para], len(para)
            else:
                current.append(para)
                current_len += len(para)
        if current:
            chunk_text = "\n".join(current)
            chunks.append({
                "text": chunk_text, "brand": brand, "model": model, "version": version,
                "section": classify_section(chunk_text),
                "page": page_idx + 1, "source_file": os.path.basename(filepath)
            })
    return chunks, brand, model, version

def embed_chunks(chunks):
    """Embeds chunks in batches with rate limit retries."""
    BATCH = 15
    TRANSIENT = ("429", "500", "503", "connection", "timeout")
    for i in range(0, len(chunks), BATCH):
        batch = chunks[i:i+BATCH]
        contents = [
            f"Car Brand: {c['brand']}\nModel: {c['model']}\nVersion: {c['version']}\nSection: {c['section']}\nPage: {c['page']}\n{c['text']}"
            for c in batch
        ]
        for attempt in range(6):
            try:
                resp = genai.embed_content(model='models/gemini-embedding-001', content=contents)
                for j, emb in enumerate(resp['embedding']):
                    batch[j]['embedding'] = np.array(emb, dtype=np.float32)
                break
            except Exception as e:
                err_lower = str(e).lower()
                if any(t in err_lower for t in TRANSIENT) and attempt < 5:
                    wait = 5 * (2 ** attempt)
                    print(f"  API/Rate error (attempt {attempt+1}/6) - retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    raise
        time.sleep(0.5)

index_data = {"files": {}, "chunks": []}

def index_pdf(filepath):
    fname = os.path.basename(filepath)
    fhash = hashlib.md5(open(filepath,'rb').read()).hexdigest()
    if fname in index_data["files"] and index_data["files"][fname].get("hash") == fhash:
        print(f"  {fname} already indexed - skipping.")
        return
    index_data["chunks"] = [c for c in index_data["chunks"] if c.get("source_file") != fname]
    print(f"  Chunking {fname}...")
    chunks, brand, model, version = chunk_pdf(filepath)
    print(f"     Got {len(chunks)} chunks. Generating embeddings...")
    embed_chunks(chunks)
    index_data["chunks"].extend(chunks)
    index_data["files"][fname] = {"hash": fhash, "brand": brand, "model": model, "version": version, "chunks_count": len(chunks)}
    print(f"  Done: {brand} {model} (version: {version}) ({len(chunks)} chunks)")

SAMPLES_BASE = "https://raw.githubusercontent.com/avanishar/drivewiseapp/main"
SAMPLE_FILES = ["Mahindra_XUV700.pdf", "Honda_Amaze.pdf", "Hundai_Creta.pdf"]

os.makedirs("brochures", exist_ok=True)
print("Downloading sample brochures...")
# Try loading GITHUB_TOKEN for private repo downloads
github_token = None
try:
    from google.colab import userdata
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not github_token:
    github_token = os.environ.get('GITHUB_TOKEN')

for fname in SAMPLE_FILES:
    fpath = f"brochures/{fname}"
    if not os.path.exists(fpath):
        url = f"{SAMPLES_BASE}/{fname}"
        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            if github_token:
                headers["Authorization"] = f"token {github_token}"
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req) as response:
                with open(fpath, 'wb') as out_file:
                    out_file.write(response.read())
            size_kb = os.path.getsize(fpath) / 1024
            if size_kb < 10:
                os.remove(fpath)
                print(f"  FAILED: {fname} too small ({size_kb:.1f} KB)")
            else:
                print(f"  Downloaded: {fname} ({size_kb/1024:.2f} MB)")
        except Exception as e:
            print(f"  Unable to download {fname}: {e}")
            print(f"  [Private Repo Note] If this repository is private, you can enable auto-downloads by adding a GITHUB_TOKEN (Personal Access Token) to the Colab Secrets panel.")
            print(f"  Alternative: Manually upload your brochure PDFs directly into the Colab file manager sidebar.")
    else:
        print(f"  Already present: {fname}")

# Auto-move any PDFs uploaded directly to the root directory into the brochures/ folder
for fname in os.listdir("."):
    if fname.lower().endswith(".pdf"):
        os.rename(fname, f"brochures/{fname}")
        print(f"  Auto-detected and moved {fname} to brochures/ folder.")

print("\nBuilding index from brochures (takes ~2 min)...")
for fname in os.listdir("brochures"):
    if fname.lower().endswith(".pdf"):
        index_pdf(f"brochures/{fname}")

print(f"\nIndex ready! {len(index_data['chunks'])} chunks from {len(index_data['files'])} car(s).")

## Step 4 — (Optional) Upload Your Own Brochure PDF

Upload any car brochure PDF. Standard naming is `Brand_Model.pdf` (e.g. `Toyota_Fortuner.pdf`). The system will parse and extract model/version metadata and update the index.

In [ ]:
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

upload_status = widgets.HTML(value='')

if IN_COLAB:
    upload_btn = widgets.Button(
        description='Upload PDF Brochure',
        button_style='info', icon='upload',
        layout=widgets.Layout(width='220px', height='40px')
    )
    def on_upload(b):
        upload_status.value = '<span style="color:#3b82f6;">Opening file picker...</span>'
        uploaded = colab_files.upload()
        if not uploaded:
            upload_status.value = '<span style="color:#64748b;">No file selected.</span>'
            return
        for filename, content in uploaded.items():
            if not filename.lower().endswith('.pdf'):
                upload_status.value = f'<span style="color:#ef4444;">Not a PDF: {filename}</span>'
                continue
            fpath = f"brochures/{filename}"
            with open(fpath, 'wb') as f:
                f.write(content)
            upload_status.value = f'<span style="color:#3b82f6;">Indexing {filename}... (~1-2 min)</span>'
            try:
                index_pdf(fpath)
                new_brands = sorted(set(m.get('brand') for m in index_data['files'].values()))
                brand_dd.options = new_brands
                upload_status.value = f'<span style="color:#10b981;">Done! {filename} indexed. Select in chat below.</span>'
            except Exception as e:
                upload_status.value = f'<span style="color:#ef4444;">Error: {e}</span>'
    upload_btn.on_click(on_upload)
    display(widgets.VBox([
        widgets.HTML(
            '<div style="background:#eff6ff;border:1px solid #bfdbfe;border-radius:10px;padding:12px 16px;">'
            '<b>Upload Your Own Car Brochure PDF</b><br>'
            '<span style="font-size:0.85rem;color:#475569;">Name it Brand_Model.pdf '
            '(e.g. Toyota_Fortuner.pdf). It will be indexed and available in the chat.</span></div>'
        ),
        upload_btn,
        upload_status
    ]))
else:
    print("Running locally: place your PDF in the brochures/ folder and re-run the indexing cell.")

## Step 5 — Two-Stage Retrieval with Generative Re-Ranking

Here we implement a distinct two-stage retrieval pipeline:
- **Stage 1 (Retrieval)**: Filters chunks by brand and model, computes a hybrid similarity score (80% semantic cosine similarity + 20% normalized keyword match), and picks the top $2 \times limit$ candidates.
- **Stage 2 (Re-Ranking)**: Feeds the candidate list into Gemini 2.5 Flash to re-rank them strictly by relevance to the query, returning the final top $limit$ chunks.

In [ ]:
STOPWORDS = {
    "a","about","above","after","again","all","am","an","and","any","are","as","at",
    "be","because","been","before","being","below","between","both","but","by","can",
    "did","do","does","doing","down","during","each","for","from","had","has","have",
    "having","he","her","here","him","his","how","i","if","in","into","is","it","its",
    "me","more","most","my","no","nor","not","of","off","on","once","only","or","other",
    "our","out","over","own","same","she","should","so","some","such","than","that",
    "the","their","them","then","there","these","they","this","those","through","to",
    "too","under","until","up","very","was","we","were","what","when","where","which",
    "while","who","whom","why","with","you","your"
}

def cosine_sim(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    return float(np.dot(v1, v2) / (n1 * n2)) if n1 > 0 and n2 > 0 else 0.0

def kw_score(query, text):
    words = [w.strip("?,.:;!\"'()").lower() for w in query.split() if w.lower() not in STOPWORDS]
    if not words: return 0.0
    tl = text.lower()
    return sum(1 for w in words if w in tl) / len(words)

def rerank_chunks(query, chunks, limit=4):
    """
    Stage 2 Re-Ranking: Uses Gemini 2.5 Flash as an LLM-based re-ranker
    to reorder retrieved candidate chunks by relevance.
    """
    if not chunks or len(chunks) <= 1:
        return chunks[:limit]
    try:
        candidates_str = ""
        for idx, c in enumerate(chunks):
            candidates_str += f"\n--- Candidate [{idx}] (Pg {c['page']}) ---\n{c['text']}\n"
            
        prompt = f"""
        You are an expert automotive search ranker. Your task is to re-rank the following candidate chunks from a car brochure based on their relevance to the user's query.
        
        User Query: "{query}"
        
        Candidate Chunks:
        {candidates_str}
        
        Re-rank these candidates from most relevant to least relevant. Return the ordered list of candidate indices as a JSON array of integers.
        For example, if candidate [2] is the most relevant, followed by [0], then [1], respond with exactly:
        [2, 0, 1]
        
        Do NOT explain your reasoning, do NOT output markdown code blocks, respond with ONLY the JSON array.
        """
        model_gen = genai.GenerativeModel("models/gemini-flash-latest")
        response = generate_content_with_retry(model_gen, prompt)
        text = response.text.strip()
        if text.startswith("```json"):
            text = text[7:]
        if text.endswith("```"):
            text = text[:-3]
        text = text.strip()
        ordered_indices = json.loads(text)
        ordered_indices = [int(i) for i in ordered_indices if 0 <= int(i) < len(chunks)]
        
        reranked = [chunks[i] for i in ordered_indices]
        # Append any missed chunks
        for c in chunks:
            if c not in reranked:
                reranked.append(c)
        return reranked[:limit]
    except Exception as e:
        print(f"  Re-ranking error, falling back: {e}")
        return chunks[:limit]

def retrieve(query, brand, model, limit=4):
    # Stage 1: Hybrid Retrieval
    filtered = [c for c in index_data['chunks']
                if c.get('brand','').lower() == brand.lower()
                and c.get('model','').lower() == model.lower()]
    if not filtered:
        return []
    emb_resp = genai.embed_content(model='models/gemini-embedding-001', content=query)
    q_emb = emb_resp['embedding']
    scored = []
    for c in filtered:
        emb = c.get('embedding')
        if emb is None: continue
        sem = cosine_sim(q_emb, emb)
        kw  = kw_score(query, c['text'])
        scored.append({**c, 'score': 0.8*sem + 0.2*kw})
    scored.sort(key=lambda x: x['score'], reverse=True)
    
    # Select double the limit as candidates for Stage 2
    candidates = scored[:limit*2]
    
    # Stage 2: Generative Re-ranking
    return rerank_chunks(query, candidates, limit)

def generate_answer(query, brand, model):
    chunks = retrieve(query, brand, model)
    if not chunks:
        return f"No brochure data found for '{brand} {model}'.", []
    context_str = "".join(
        f"\n--- Source [{i+1}] (Page {c['page']}, Section: {c['section']}, Version: {c.get('version', '1.0')}) ---\n{c['text']}\n"
        for i, c in enumerate(chunks)
    )
    sys_prompt = (
        f"You are an expert automotive assistant for Drive Wise. "
        f"Answer ONLY from the brochure excerpts for {brand} {model}. "
        "Rules: 1) Use only the provided context. 2) If missing, say so. "
        "3) Use inline citations [1],[2] matching source numbers. 4) Be clear and concise."
    )
    prompt = f"Brochure Context for {brand} {model}:\n{context_str}\nUser Query: \"{query}\"\nGrounded Answer:"
    try:
        model_gen = genai.GenerativeModel(
            model_name="models/gemini-flash-latest",
            system_instruction=sys_prompt
        )
        resp = generate_content_with_retry(model_gen, prompt, generation_config=genai.types.GenerationConfig(temperature=0.1))
        return resp.text.strip(), chunks
    except Exception as e:
        return f"Error: {e}", []

def get_car_map():
    cm = {}
    for meta in index_data['files'].values():
        b, m = meta.get('brand','Unknown'), meta.get('model','Unknown')
        if b not in cm: cm[b] = []
        if m not in cm[b]: cm[b].append(m)
    return cm

print("RAG engine and Re-Ranker ready!")

## Step 6 — Interactive Chat UI with Evaluation Metrics & Logging

This cell sets up an interactive chat interface inside the notebook. Every query triggers:
1. **Quality Evaluation**: Runs LLM-as-a-judge scoring for **Faithfulness**, **Context Relevance**, and **Answer Correctness** on a 1-5 scale.
2. **Persistent Logging**: Writes the query, answer, source metadata (including versions), latencies, failure flags, and quality scores into a local database (`logs/notebook_query_logs.db`).

In [ ]:
import sqlite3
from datetime import datetime

DB_PATH = "logs/notebook_query_logs.db"

def init_db():
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS query_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT,
            query TEXT,
            brand TEXT,
            model TEXT,
            response TEXT,
            response_time REAL,
            is_failed INTEGER,
            faithfulness REAL,
            context_relevance REAL,
            answer_correctness REAL,
            retrieved_chunks TEXT
        )
    """)
    conn.commit()
    conn.close()

def evaluate_answer(query, chunks, answer):
    """LLM-as-a-judge evaluation."""
    if not chunks or not answer:
        return 1.0, 1.0, 1.0, "Empty chunks or response."
    try:
        context_str = ""
        for idx, c in enumerate(chunks):
            context_str += f"\nSource [{idx+1}]: {c['text']}\n"
            
        eval_prompt = f"""
        You are a RAG quality evaluation judge. Your task is to evaluate the quality of the retrieval and generation system.
        
        Inputs to evaluate:
        - Query: "{query}"
        - Retrieved Context Chunks:
        {context_str}
        - Generated Answer: "{answer}"
        
        Evaluate and score the following metrics on a scale from 1.0 (worst) to 5.0 (best):
        1. Context Relevance: How relevant and helpful are the retrieved chunks to the user's specific query?
        2. Faithfulness: Is the generated answer completely grounded in the retrieved chunks? Award a 1.0 if it makes claims not supported by the context or has hallucinations. Award a 5.0 if it is 100% grounded.
        3. Answer Correctness & Completeness: Does the generated answer accurately and completely resolve the user query based strictly on the retrieved context?
        
        Respond with ONLY a valid JSON object in this format (no markdown blocks, just raw JSON text):
        {{
          "context_relevance": float,
          "faithfulness": float,
          "answer_correctness": float,
          "rationale": "string explanation of scores"
        }}
        """
        model_eval = genai.GenerativeModel("models/gemini-flash-latest")
        response = generate_content_with_retry(model_eval, eval_prompt)
        text = response.text.strip()
        if text.startswith("```json"):
            text = text[7:]
        if text.endswith("```"):
            text = text[:-3]
        text = text.strip()
        data = json.loads(text)
        return (
            float(data.get("context_relevance", 1.0)),
            float(data.get("faithfulness", 1.0)),
            float(data.get("answer_correctness", 1.0)),
            data.get("rationale", "")
        )
    except Exception as e:
        print(f"  Evaluation error: {e}")
        return 1.0, 1.0, 1.0, f"Error: {e}"

def log_to_db(query, brand, model, answer, elapsed, is_failed, faithfulness, relevance, correctness, chunks):
    init_db()
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    timestamp = datetime.now().isoformat()
    chunks_meta = [
        {"page": c["page"], "section": c["section"], "version": c.get("version", "1.0"), "source_file": c["source_file"]}
        for c in chunks
    ]
    cursor.execute("""
        INSERT INTO query_logs (
            timestamp, query, brand, model, response, response_time, 
            is_failed, faithfulness, context_relevance, answer_correctness, retrieved_chunks
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        timestamp, query, brand, model, answer, elapsed, 
        int(is_failed), faithfulness, relevance, correctness, json.dumps(chunks_meta)
    ))
    conn.commit()
    conn.close()

chat_history = []
CAR_MAP = get_car_map()
brand_list = sorted(CAR_MAP.keys())

brand_dd = widgets.Dropdown(
    options=brand_list,
    value=brand_list[0] if brand_list else None,
    description='Brand:',
    style={'description_width': '55px'},
    layout=widgets.Layout(width='220px')
)
model_dd = widgets.Dropdown(
    options=CAR_MAP.get(brand_list[0], []) if brand_list else [],
    description='Model:',
    style={'description_width': '55px'},
    layout=widgets.Layout(width='220px')
)
query_box = widgets.Textarea(
    placeholder='Type your question here...\nExamples:\n- What safety features does it have?\n- What is the mileage?\n- What is the document version of this brochure?',
    layout=widgets.Layout(width='100%', height='90px')
)
ask_btn = widgets.Button(
    description='Ask DriveWise',
    button_style='primary', icon='search',
    layout=widgets.Layout(width='165px', height='38px')
)
clear_btn = widgets.Button(
    description='Clear Chat',
    button_style='warning', icon='trash',
    layout=widgets.Layout(width='130px', height='38px')
)
status_lbl = widgets.HTML(
    value='<span style="color:#64748b;font-size:0.88rem;">Ready! Select a car and ask a question.</span>'
)
chat_out = widgets.Output(layout=widgets.Layout(
    border='1px solid #e2e8f0', border_radius='10px', padding='14px',
    min_height='180px', max_height='520px', overflow_y='auto'
))

def on_brand_change(change):
    model_dd.options = get_car_map().get(change['new'], [])

brand_dd.observe(on_brand_change, names='value')

def render_chat():
    with chat_out:
        clear_output(wait=True)
        if not chat_history:
            display(HTML(
                '<div style="text-align:center;padding:36px;color:#94a3b8;">'
                '<div style="font-size:3rem;">&#x1F697;</div>'
                '<div style="font-size:1.05rem;margin-top:8px;font-weight:600;">Ask anything about your car!</div>'
                '<div style="font-size:0.82rem;margin-top:6px;color:#cbd5e1;">'
                'mileage &bull; safety features &bull; engine specs &bull; document versions &bull; re-ranked chunks'
                '</div></div>'
            ))
            return
        for item in chat_history:
            display(HTML(
                '<div style="margin:10px 0;display:flex;justify-content:flex-end;">'
                '<div style="background:#eff6ff;border:1px solid #bfdbfe;border-radius:16px 16px 4px 16px;'
                'padding:10px 14px;max-width:80%;font-size:0.92rem;color:#1e3a5f;">'
                f'<b>You</b> <span style="font-size:0.75rem;color:#94a3b8;">({item["car"]})</span><br>'
                f'{item["query"]}</div></div>'
            ))
            answer_html = item['answer'].replace('\n', '<br>')
            display(HTML(
                '<div style="margin:6px 0 12px 0;">'
                '<div style="background:#fff;border:1px solid #e2e8f0;border-left:4px solid #3b82f6;'
                'border-radius:4px 14px 14px 14px;padding:12px 16px;max-width:95%;'
                'font-size:0.92rem;color:#1e293b;box-shadow:0 2px 6px rgba(0,0,0,0.05);">'
                f'<b>Drive Wise</b> <span style="font-size:0.75rem;color:#64748b;">({item["time"]}s)</span><br><br>'
                f'{answer_html}</div></div>'
            ))
            if item.get('sources'):
                tags = "".join(
                    f'<span style="display:inline-block;background:#f0fdf4;color:#166534;'
                    f'font-size:0.73rem;padding:2px 8px;border-radius:20px;margin:2px;'
                    f'border:1px solid #bbf7d0;">[{i+1}] Pg {s["page"]} - {s["section"]} (Ver: {s.get("version", "1.0")})</span>'
                    for i, s in enumerate(item['sources'])
                )
                display(HTML(f'<div style="margin:-4px 0 10px 12px;font-size:0.8rem;">Sources: {tags}</div>'))
            if item.get('metrics'):
                m = item['metrics']
                display(HTML(
                    f'<div style="margin:-4px 0 10px 12px;font-size:0.8rem;color:#475569;background:#f8fafc;padding:6px 10px;border-radius:8px;">'
                    f'📊 <b>LLM-as-a-Judge Evaluation:</b> &nbsp; '
                    f'Faithfulness: <b>{m["faithfulness"]:.1f}/5.0</b> &bull; '
                    f'Context Relevance: <b>{m["relevance"]:.1f}/5.0</b> &bull; '
                    f'Answer Correctness: <b>{m["correctness"]:.1f}/5.0</b><br>'
                    f'<span style="font-size:0.74rem;color:#64748b;"><b>Judge Rationale:</b> {m["rationale"]}</span></div>'
                ))
            display(HTML("<hr style='border:none;border-top:1px solid #f1f5f9;margin:4px 0;'>"))

def on_ask(b):
    query = query_box.value.strip()
    if not query:
        status_lbl.value = '<span style="color:#ef4444;">Please type a question first!</span>'
        return
    if not brand_dd.value or not model_dd.value:
        status_lbl.value = '<span style="color:#ef4444;">No car indexed yet. Run Step 3 first!</span>'
        return
    brand, model = brand_dd.value, model_dd.value
    ask_btn.disabled = clear_btn.disabled = True
    ask_btn.description = 'Thinking...'
    status_lbl.value = f'<span style="color:#3b82f6;">Searching, re-ranking, and generating answer...</span>'
    query_box.value = ''
    t0 = time.time()
    answer, sources = generate_answer(query, brand, model)
    elapsed = round(time.time() - t0, 2)
    
    is_failed = False
    if not sources or any(w in answer.lower() for w in ["sorry", "not available", "error"]):
        is_failed = True
        
    status_lbl.value = f'<span style="color:#3b82f6;">Evaluating quality metrics...</span>'
    relevance, faithfulness, correctness, rationale = evaluate_answer(query, sources, answer)
    
    log_to_db(query, brand, model, answer, elapsed, is_failed, faithfulness, relevance, correctness, sources)
    
    metrics = {
        "relevance": relevance,
        "faithfulness": faithfulness,
        "correctness": correctness,
        "rationale": rationale
    }
    chat_history.append({'query': query, 'answer': answer, 'sources': sources,
                         'car': f'{brand} {model}', 'time': elapsed, 'is_failed': is_failed, 'metrics': metrics})
    render_chat()
    status_lbl.value = f'<span style="color:#10b981;">Done in {elapsed}s - quality score: {(relevance+faithfulness+correctness)/3:.2f}/5.0</span>'
    ask_btn.disabled = clear_btn.disabled = False
    ask_btn.description = 'Ask DriveWise'

def on_clear(b):
    chat_history.clear()
    render_chat()
    status_lbl.value = '<span style="color:#64748b;">Chat cleared.</span>'

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)

render_chat()
display(widgets.VBox([
    widgets.HTML(
        '<div style="background:linear-gradient(135deg,#1e3a8a,#3b82f6);color:white;'
        'padding:14px 20px;border-radius:12px 12px 0 0;">'
        '<span style="font-size:1.2rem;font-weight:700;">Drive Wise - Interactive Chat</span>'
        '<span style="font-size:0.82rem;opacity:0.8;margin-left:10px;">Grounded answers from car brochures</span>'
        '</div>'
    ),
    widgets.HBox([brand_dd, model_dd], layout=widgets.Layout(gap='10px', margin='8px 0')),
    widgets.HTML('<div style="font-weight:600;color:#334155;margin:4px 0 2px 0;">Your Question:</div>'),
    query_box,
    widgets.HBox([ask_btn, clear_btn, status_lbl],
                 layout=widgets.Layout(gap='10px', align_items='center', margin='6px 0')),
    chat_out
], layout=widgets.Layout(
    border='1px solid #e2e8f0', border_radius='12px',
    padding='16px', width='100%'
)))

## Step 7 — Evaluation & Quality Monitoring Dashboard

Reads from the persistent SQLite database (`logs/notebook_query_logs.db`) and aggregates metrics over time, displaying graphs for average quality scores, latencies, query counts, and failure rates.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def show_dashboard():
    if not os.path.exists(DB_PATH):
        print("No query log database found. Ask some questions in the chat above first!")
        return
        
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM query_logs", conn)
    conn.close()
    
    if df.empty:
        print("Query log database is empty. Send a few messages in the chat to populate dashboard.")
        return
        
    # Aggregate performance statistics
    total_queries = len(df)
    avg_latency = df['response_time'].mean()
    failures = df['is_failed'].sum()
    failure_rate = (failures / total_queries) * 100.0
    avg_faithfulness = df['faithfulness'].mean()
    avg_relevance = df['context_relevance'].mean()
    avg_correctness = df['answer_correctness'].mean()
    avg_quality = (avg_faithfulness + avg_relevance + avg_correctness) / 3
    failure_color = '#ef4444' if failure_rate > 10 else '#10b981'
    
    summary_html = """
    <div style="display: flex; gap: 15px; margin-bottom: 20px;">
        <div style="flex: 1; padding: 12px 18px; border-radius: 10px; background: #f8fafc; border: 1px solid #e2e8f0; text-align: center;">
            <div style="font-size: 1.8rem; font-weight: 700; color: #1e3a8a;">{total_queries}</div>
            <div style="font-size: 0.8rem; color: #64748b; text-transform: uppercase; font-weight: 600;">Total Queries</div>
        </div>
        <div style="flex: 1; padding: 12px 18px; border-radius: 10px; background: #f8fafc; border: 1px solid #e2e8f0; text-align: center;">
            <div style="font-size: 1.8rem; font-weight: 700; color: #1e3a8a;">{avg_latency:.2f}s</div>
            <div style="font-size: 0.8rem; color: #64748b; text-transform: uppercase; font-weight: 600;">Avg Latency</div>
        </div>
        <div style="flex: 1; padding: 12px 18px; border-radius: 10px; background: #f8fafc; border: 1px solid #e2e8f0; text-align: center;">
            <div style="font-size: 1.8rem; font-weight: 700; color: {failure_color};"><span style='color: {failure_color};'>{failure_rate:.1f}%</span></div>
            <div style="font-size: 0.8rem; color: #64748b; text-transform: uppercase; font-weight: 600;">Failure Rate</div>
        </div>
        <div style="flex: 1; padding: 12px 18px; border-radius: 10px; background: #f8fafc; border: 1px solid #e2e8f0; text-align: center;">
            <div style="font-size: 1.8rem; font-weight: 700; color: #0d9488;">{avg_quality:.2f} / 5.0</div>
            <div style="font-size: 0.8rem; color: #64748b; text-transform: uppercase; font-weight: 600;">Quality Score</div>
        </div>
    </div>
    """.format(
        total_queries=total_queries,
        avg_latency=avg_latency,
        failure_color=failure_color,
        failure_rate=failure_rate,
        avg_quality=avg_quality
    )
    display(HTML(summary_html))
    
    # Generate plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Quality Metrics Plot
    labels = ['Faithfulness', 'Context Relevance', 'Answer Correctness']
    values = [avg_faithfulness, avg_relevance, avg_correctness]
    axes[0].bar(labels, values, color=['#2563eb', '#059669', '#d97706'], width=0.4)
    axes[0].set_ylim(0, 5.5)
    axes[0].set_title('Average RAG Quality (LLM-Judge)')
    axes[0].set_ylabel('Score (out of 5)')
    for idx, val in enumerate(values):
        axes[0].text(idx, val + 0.15, f"{val:.2f}", ha='center', fontweight='bold')
        
    # 2. Queries by car model
    counts = df.groupby(['brand', 'model']).size().reset_index(name='count')
    car_labels = [f"{r['brand']} {r['model']}" for _, r in counts.iterrows()]
    axes[1].pie(counts['count'], labels=car_labels, autopct='%1.1f%%', startangle=140, colors=['#93c5fd', '#a7f3d0', '#fef08a', '#fbcfe8'])
    axes[1].set_title('Query Distribution by Car model')
    
    plt.tight_layout()
    plt.show()
    
    print("\n--- Recent 10 Persistent Log Entries ---")
    recent_df = df[['timestamp', 'brand', 'model', 'query', 'response_time', 'is_failed', 'faithfulness', 'context_relevance', 'answer_correctness']].tail(10)
    display(recent_df)

show_dashboard()

## Step 8 — Launching Standalone Streamlit Web Interface

In addition to the in-notebook UI, DriveWise includes a beautiful standalone **Streamlit Web Application** (`app.py`).

### How to Run Locally
If you are running this code on your local system, open a terminal in the folder and run:
```bash
streamlit run app.py
```

### How to Expose Web Interface on Google Colab
If you are running inside Google Colab, execute the cell below. It will install `localtunnel` and start the Streamlit server. Once started, click the tunnel link and enter your external IP address as the password to access your web interface live!

In [ ]:
# Fetch the password IP address first
import urllib.request
print("1. Fetching external IP address for localtunnel password...")
try:
    external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print(f"   PASSWORD IP: {external_ip}   <-- Copy this!")
except Exception as e:
    print(f"   Failed to get external IP: {e}")

print("\n2. Starting localtunnel and Streamlit (this will run continuously until stopped)...\n")
!npm install -g localtunnel
!streamlit run app.py & npx localtunnel --port 8501